In [ ]:
#| include: false
import pandas as pd
import numpy as np
import plotly.io as pio
from IPython.display import display, Markdown

pio.renderers.default = 'notebook'
pio.templates.default = 'plotly_white'

from aavolve.report_helpers import *

# parameters (overridden by papermill)
group_id = 'GROUP'
manifest = 'out/qc/group_reports/GROUP_manifest.tsv'


<script type="text/javascript">
function aavolveResizePlotly(container) {
  if (typeof require === "undefined") return;
  require(["plotly"], function(Plotly) {
    (container || document).querySelectorAll(".plotly-graph-div").forEach(function(gd) {
      try { Plotly.Plots.resize(gd); } catch (e) {}
    });
  });
}
document.addEventListener("shown.bs.tab", function(e) {
  var selector = e.target && e.target.getAttribute && e.target.getAttribute("data-bs-target");
  var pane = selector ? document.querySelector(selector) : null;
  aavolveResizePlotly(pane);
});
window.addEventListener("resize", function() { aavolveResizePlotly(); });
</script>


In [ ]:
#| include: false
manifest_df = pd.read_csv(manifest, sep='	')
samples = manifest_df['sample'].tolist()
parent_file = manifest_df['parent_file'].iloc[0]
reference_file = manifest_df['reference_file'].iloc[0]
color_map = parent_colors(manifest_df['parent_frequencies'].iloc[0])


In [ ]:
display(Markdown('This report aggregates all samples which share the same parent/reference combination.'))
display(Markdown(f'- Group ID: `{group_id}`'))
display(Markdown(f'- Parent file: `{parent_file}`'))
display(Markdown(f'- Reference file: `{reference_file}`'))
display(Markdown(f"- Samples: `{', '.join(samples)}`"))


In [ ]:
#| title: Read counts at each processing stage (all samples)

dfs = []
for row in manifest_df.itertuples(index=False):
    df = import_read_count_data(row.read_counts, row.seq_tech)
    df = df[df['File type'] != 'Distinct at nucleotide level']
    df = df[df['File type'] != 'Distinct at amino acid level']
    df['sample'] = row.sample
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)

fig = px.line(df_all, x='File type', y='Count', color='sample', markers=True)
fig.update_xaxes(tickangle=90, automargin=True)
fig.update_layout(
    margin=dict(l=60, r=40, t=60, b=260),
    legend=dict(
        title_text='Sample',
        orientation='h',
        entrywidth=140,
        entrywidthmode='pixels',
        x=0,
        xanchor='left',
        y=-0.45,
        yanchor='top',
        font=dict(size=10),
    ),
)
fig

In [ ]:
#| title: Fraction of reads retained after each filter (all samples)

fig = px.line(df_all, x='File type', y='Fraction of reads', color='sample', markers=True)
fig.update_xaxes(tickangle=90, automargin=True)
fig.update_layout(
    margin=dict(l=60, r=40, t=60, b=260),
    legend=dict(
        title_text='Sample',
        orientation='h',
        entrywidth=140,
        entrywidthmode='pixels',
        x=0,
        xanchor='left',
        y=-0.45,
        yanchor='top',
        font=dict(size=10),
    ),
)
fig

In [ ]:
#| title: Overall parent assignment summary (mean across variants)

summary_rows = []
for row in manifest_df.itertuples(index=False):
    df = pd.read_csv(row.parent_frequencies, sep='	')
    df['parent'] = df['parent'].astype(str).str.replace('non_parental_\d+', 'non parental', regex=True)
    df_sum = df.groupby('parent', as_index=False)['frequency'].mean()
    df_sum['frequency'] = df_sum['frequency'] * 100
    df_sum['sample'] = row.sample
    summary_rows.append(df_sum)
summary_df = pd.concat(summary_rows, ignore_index=True)

fig = px.bar(
    summary_df,
    x='parent',
    y='frequency',
    color='parent',
    facet_col='sample',
    facet_col_wrap=2,
    labels={'frequency': 'Mean parent frequency (%)', 'parent': 'Parent'},
    color_discrete_map=color_map,
)
fig.update_xaxes(tickangle=90, automargin=True)
fig.update_layout(margin=dict(l=60, r=40, t=60, b=160), showlegend=False)
fig

In [ ]:
#| title: Assigned parents for top reads (per sample)
#| fig-width: 100%

for row in manifest_df.itertuples(index=False):
    display(Markdown(f'## {row.sample}'))
    fig = parent_heatmap(row.assigned_parents, row.parent_frequencies)
    if fig is not None:
        fig.show()

In [ ]:
#| title: Mean pairwise distance between top capsids

metric_rows = []
for row in manifest_df.itertuples(index=False):
    for label, path in [
        ('nt-first', row.dmat_nt_first),
        ('aa-first', row.dmat_aa_first),
    ]:
        dmat = np.loadtxt(path)
        mask = ~np.eye(dmat.shape[0], dtype=bool)
        metric_rows.append({'sample': row.sample, 'matrix': label, 'mean_distance': float(dmat[mask].mean())})
metrics_df = pd.DataFrame(metric_rows)

fig = px.bar(metrics_df, x='sample', y='mean_distance', color='matrix', barmode='group',
             labels={'mean_distance': 'Mean pairwise distance', 'sample': 'Sample', 'matrix': 'Matrix'})
fig.update_layout(margin=dict(l=60, r=40, t=60, b=80))
fig